In [1]:
import pandas as pd


In [1]:
import os

print(os.listdir("../data"))
print(os.listdir("../data/archive (9)"))
print(os.listdir("../data/archive (5)"))


['archive (5)', 'archive (5).zip', 'archive (9)', 'archive (9).zip']
['test.csv', 'training.csv', 'validation.csv']
['dataset.csv']


In [2]:
import pandas as pd

train = pd.read_csv("../data/archive (9)/training.csv")
val   = pd.read_csv("../data/archive (9)/validation.csv")
test  = pd.read_csv("../data/archive (9)/test.csv")

emotion_df = pd.concat([train, val, test], ignore_index=True)

emotion_df.head()


,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [3]:
emotion_df.columns


Index(['text', 'label'], dtype='object')

In [4]:
mental_df = pd.read_csv("../data/archive (5)/dataset.csv")
mental_df.head()


,title,content,score
0,"Regular check-in post, with information about ...",Welcome to /r/depression's check-in post - a p...,115
1,Our most-broken and least-understood rules is ...,We understand that most people who reply immed...,2365
2,Going back to college at 33 after 3 times of d...,"I've always wanted to go back to school, the c...",84
3,Crying alone all the time,All I wan is to be loved. I just want someone...,108
4,I genuinely don’t think I’ll ever be able to h...,"I don’t even drive, didn’t finish my high scho...",60


In [5]:
mental_df.columns


Index(['title', 'content', 'score'], dtype='object')

In [6]:
mental_df["text"] = mental_df["content"]


In [7]:
def score_to_stress(score):
    if score <= 4:
        return "Low"
    elif score <= 7:
        return "Medium"
    else:
        return "High"

mental_df["stress_level"] = mental_df["score"].apply(score_to_stress)


In [8]:
mental_df["emotion"] = "Unknown"


In [9]:
mental_final = mental_df[["text", "stress_level", "emotion"]]
mental_final.head()


,text,stress_level,emotion
0,Welcome to /r/depression's check-in post - a p...,High,Unknown
1,We understand that most people who reply immed...,High,Unknown
2,"I've always wanted to go back to school, the c...",High,Unknown
3,All I wan is to be loved. I just want someone...,High,Unknown
4,"I don’t even drive, didn’t finish my high scho...",High,Unknown


In [10]:
emotion_final = emotion_df[["text", "stress_level", "emotion"]]

final_df = pd.concat([mental_final, emotion_final], ignore_index=True)

final_df.dropna(inplace=True)
final_df = final_df[final_df["text"].str.len() > 10]

final_df.head()


KeyError: "['stress_level', 'emotion'] not in index"

In [11]:
emotion_df.columns


Index(['text', 'label'], dtype='object')

In [12]:
emotion_map = {
    "joy": "Happy",
    "sadness": "Sad",
    "anger": "Angry",
    "fear": "Fear",
    "neutral": "Neutral"
}

emotion_df["emotion"] = emotion_df["label"].map(emotion_map)


In [13]:
emotion_df.head()


,text,label,emotion
0,i didnt feel humiliated,0,NaN
1,i can go from feeling so hopeless to so damned...,0,NaN
2,im grabbing a minute to post i feel greedy wrong,3,NaN
3,i am ever feeling nostalgic about the fireplac...,2,NaN
4,i am feeling grouchy,3,NaN


In [14]:
emotion_df["label"].value_counts()


label
1    6761
0    5797
3    2709
4    2373
2    1641
5     719
Name: count, dtype: int64

In [15]:
emotion_map = {
    0: "Sad",
    1: "Happy",
    2: "Love",
    3: "Angry",
    4: "Fear",
    5: "Surprise"
}

emotion_df["emotion"] = emotion_df["label"].map(emotion_map)


In [16]:
emotion_df.head()
emotion_df["emotion"].value_counts()


emotion
Happy       6761
Sad         5797
Angry       2709
Fear        2373
Love        1641
Surprise     719
Name: count, dtype: int64

In [17]:
def emotion_to_stress(emotion):
    if emotion in ["Happy", "Love", "Surprise"]:
        return "Low"
    elif emotion == "Sad":
        return "Medium"
    else:  # Angry, Fear
        return "High"

emotion_df["stress_level"] = emotion_df["emotion"].apply(emotion_to_stress)


In [18]:
emotion_df["stress_level"].value_counts()


stress_level
Low       9121
Medium    5797
High      5082
Name: count, dtype: int64

In [19]:
emotion_final = emotion_df[["text", "stress_level", "emotion"]]
emotion_final.head()


,text,stress_level,emotion
0,i didnt feel humiliated,Medium,Sad
1,i can go from feeling so hopeless to so damned...,Medium,Sad
2,im grabbing a minute to post i feel greedy wrong,High,Angry
3,i am ever feeling nostalgic about the fireplac...,Low,Love
4,i am feeling grouchy,High,Angry


In [20]:
final_df = pd.concat([mental_final, emotion_final], ignore_index=True)

final_df.dropna(inplace=True)
final_df = final_df[final_df["text"].str.len() > 10]

final_df.head()


,text,stress_level,emotion
0,Welcome to /r/depression's check-in post - a p...,High,Unknown
1,We understand that most people who reply immed...,High,Unknown
2,"I've always wanted to go back to school, the c...",High,Unknown
3,All I wan is to be loved. I just want someone...,High,Unknown
4,"I don’t even drive, didn’t finish my high scho...",High,Unknown


In [21]:
final_df.shape


(31500, 3)

In [22]:
final_df["stress_level"].value_counts()


stress_level
Low       16694
High       7625
Medium     7181
Name: count, dtype: int64

In [23]:
final_df["emotion"].value_counts()


emotion
Unknown     11504
Happy        6760
Sad          5796
Angry        2708
Fear         2372
Love         1641
Surprise      719
Name: count, dtype: int64

In [24]:
final_df.to_csv("../data/final_mental_health_dataset.csv", index=False)
